In [8]:
import re
import nltk
import pandas as pd
import numpy as np

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [33]:
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/priyanka/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/priyanka/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Keep negation terms because they are highly informative for sentiment.
negation_words = {'no', 'not', 'nor', 'never'}
custom_stop_words = stop_words - negation_words

def preprocess(text):
    text = re.sub('[^a-zA-Z]', ' ', str(text))
    text = text.lower()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in custom_stop_words]
    return ' '.join(words)

In [3]:
df = pd.read_csv('all_kindle_review.csv')


df = df[df['rating'] != 3].copy()
df['clean_review'] = df['reviewText'].apply(preprocess)
df['sentiment'] = df['rating'].apply(lambda x: 1 if x >= 4 else 0)
print('Dataset shape after removing 3-star reviews:', df.shape)

Dataset shape after removing 3-star reviews: (10000, 13)


In [10]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime,clean_review,sentiment
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400,great short read want put read one sitting sex...,1
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000,not expect type book library pleased find pric...,1
5,5,3744,B0021L9YDK,"[6, 6]",5,Aislinn is a little girl with big dreams. Afte...,"12 7, 2009",A3J5NN6MJK4M4A,"Aubrie A. Dionne ""Fantasy, Sci Fi Author""",A story of a little girl with big dreams.,1260144000,aislinn little girl big dream death older brot...,1
6,6,13641,B0038NN38W,"[1, 1]",2,This has the makings of a good story... unfort...,"08 18, 2011",A531QY5K7JVXI,Chicano,This story has potential but ultimately disapp...,1313625600,making good story unfortunately disappoints te...,0
7,7,4448,B002AJ7X2C,"[1, 1]",4,I got this because I like collaborated short s...,"03 8, 2010",AN8ELR6AHMMQ,"Jessss ""I read to find stories that inspire m...",Good thriller,1268006400,got like collaborated short story alot time tw...,1


In [4]:
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 3),
    min_df=2,
    sublinear_tf=True,
    strip_accents='unicode'
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

In [6]:
param_grid = {
    'C': [1, 2, 3, 4, 5],
    'class_weight': [None, 'balanced']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=5000, solver='liblinear'),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [9]:
grid.fit(X_train_vec, y_train)
model = grid.best_estimator_
print('Best Params:', grid.best_params_)
print('Best CV Accuracy:', grid.best_score_)

Best Params: {'C': 4, 'class_weight': 'balanced'}
Best CV Accuracy: 0.9033749999999999


In [40]:
y_pred = model.predict(X_test_vec)

print('Test Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Test Accuracy: 0.9085
              precision    recall  f1-score   support

           0       0.87      0.91      0.89       800
           1       0.93      0.91      0.92      1200

    accuracy                           0.91      2000
   macro avg       0.90      0.91      0.91      2000
weighted avg       0.91      0.91      0.91      2000



In [ ]:
def predict_review(review):
    review_clean = preprocess(review)
    review_vec = tfidf.transform([review_clean])
    pred = model.predict(review_vec)[0]
    prob = model.predict_proba(review_vec)

    print("Review:", review)
    print("Prediction:", "Customer is satisfied" if pred == 1 else "Customer is not satisfied")
    print("Confidence of prediction:", prob, "\n")

In [ ]:
predict_review("best book")
predict_review("Not good, very boring and disappointing")
predict_review("This is the best book I have ever read and I loved it")
predict_review("Worst book. Total waste of money")

Review: best book
Prediction: Customer is satisfied
Confidence: [[0.31686784 0.68313216]]
Review: Not good, very boring and disappointing
Prediction: Customer is not satisfied
Confidence: [[9.99443982e-01 5.56017762e-04]]
Review: This is the best book I have ever read and I loved it
Prediction: Customer is satisfied
Confidence: [[0.246514 0.753486]]
Review: Worst book. Total waste of money
Prediction: Customer is not satisfied
Confidence: [[0.98415826 0.01584174]]
